<a href="https://colab.research.google.com/github/pradeep-999ai/research-paper-qa/blob/main/Research_Paper_QA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Paper Q&A (RAG) — Colab Version

A Retrieval-Augmented Generation notebook that answers questions about an uploaded research paper, grounded in the paper's actual text with page citations.

**How to use:**
1. Run each cell in order (Shift+Enter)
2. When asked, paste your free Gemini API key (get one at https://aistudio.google.com/app/apikey — no billing needed)
3. Upload a PDF research paper when prompted
4. Ask questions in the last cell

Repo: https://github.com/pradeep-999ai/research-paper-qa

## 1. Install dependencies

In [1]:
!pip install -q pymupdf langchain langchain-community langchain-text-splitters langchain-google-genai google-generativeai faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==

## 2. Set your Gemini API key
Get a free key (no credit card needed) at https://aistudio.google.com/app/apikey

In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

Paste your Gemini API key: ··········


## 3. PDF ingestion — extract and chunk text

In [3]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_pdf_chunks(path, chunk_size=1000, chunk_overlap=100):
    """Reads a PDF and returns a list of {'text': ..., 'page': ...} chunks."""
    doc = fitz.open(path)
    pages = [(i + 1, page.get_text()) for i, page in enumerate(doc)]
    doc.close()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )

    chunks = []
    for page_num, text in pages:
        if not text.strip():
            continue
        for piece in splitter.split_text(text):
            chunks.append({"text": piece, "page": page_num})
    return chunks

## 4. Build embeddings + FAISS index

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

def build_index(chunks, embedding_model="models/gemini-embedding-001"):
    embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model)
    texts = [c["text"] for c in chunks]
    metadatas = [{"page": c["page"]} for c in chunks]
    return FAISS.from_texts(texts, embeddings, metadatas=metadatas)

/tmp/ipykernel_3125/2976564108.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 5. Retrieval + grounded generation

**Note:** Google renames/retires Gemini model IDs fairly often. If you get a 404 "model not found" error below, the error message itself names the current replacement model — just update `model=` in the function call.

In [5]:
import google.generativeai as genai

genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

def build_prompt(question, docs):
    context = "\n\n".join(
        f"[Page {d.metadata['page']}] {d.page_content}" for d in docs
    )
    return f"""Answer ONLY using the context below.
If the answer is not in the context, say "This is not covered in the paper."

Context:
{context}

Question: {question}
Answer:"""

def answer_question(index, question, k=4, model="gemini-3.5-flash-lite"):
    docs = index.similarity_search(question, k=k)
    if not docs:
        return "This is not covered in the paper.", set()

    prompt = build_prompt(question, docs)
    gemini_model = genai.GenerativeModel(model)
    response = gemini_model.generate_content(prompt)

    pages = {d.metadata["page"] for d in docs}
    return response.text, pages

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 6. Upload a research paper (PDF)

In [6]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}")

Saving 2311.11250v1_copy.pdf to 2311.11250v1_copy.pdf
Uploaded: 2311.11250v1_copy.pdf


## 7. Index the paper

In [7]:
chunks = load_pdf_chunks(pdf_path)
print(f"Extracted {len(chunks)} chunks")

index = build_index(chunks)
print("Index built. Ready for questions!")

Extracted 93 chunks
Index built. Ready for questions!


## 8. Ask questions
Run this cell as many times as you like with different questions.

In [9]:
question = input("Ask a question about the paper: ")
answer, pages = answer_question(index, question)

print("\nAnswer:")
print(answer)
if pages:
    print(f"\nSource page(s): {sorted(pages)}")

Ask a question about the paper: What are the different levels of sentiment analysis?

Answer:
There are three levels of sentiment analysis: document level, sentence level, and aspect level.

Source page(s): [2, 6, 10]
